# Raw Data Ingestion

## Imports

In [0]:
import requests as r
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StringType
import json
import os

## Cluster enviroment keys

In [0]:
url_request = os.getenv("API_URL")
storage_account_key = os.getenv("STORAGE_ACCOUNT_KEY")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
container_raw = os.getenv("CONTAINER_RAW")

# Adls connection

In [0]:
spark.conf.set(
     f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
     storage_account_key
)

# API Call to ingest raw JSON file

In [0]:
response = r.get(url_request)

# Transforming JSON into a spark df with a single collumn with everything

In [0]:
raw_json = json.dumps(response.json())

In [0]:
raw_data_rdd = sc.parallelize([(raw_json,)])

In [0]:
df_raw = spark.createDataFrame(raw_data_rdd, ['raw_content'])

In [0]:
display(df_raw)

In [0]:
dbutils.fs.ls(f"abfss://{container_raw}@{storage_account_name}.dfs.core.windows.net/")

In [0]:
path=f"abfss://{container_raw}@{storage_account_name}.dfs.core.windows.net/"

# Saving the raw data as a spark df in the adls container

In [0]:
df_raw.coalesce(1) \
    .write \
    .mode("overwrite") \
    .json(path)